# Ghép ảnh toàn cảnh — Image Stitching
## Bài tập lớn Computer Vision, mục 2.2

Notebook này hiện thực pipeline ghép bốn ảnh có vùng chồng lấn thành panorama. SIFT và ORB được chạy trên cùng một bộ ảnh và qua cùng một pipeline. Toàn bộ mã nguồn được giữ theo từng module trong các cell bên dưới; không cần thư mục src.

**Dữ liệu mặc định:** input/boat/01.jpg–04.jpg.  
**Kết quả:** output/notebook/sift/, output/notebook/orb/ và evaluation.csv.

## 1. Bài toán, đầu vào và giả định

- Đầu vào là 3–4 ảnh màu của cùng cảnh, chụp từ vị trí quan sát gần như cố định với góc xoay khác nhau.
- Ảnh được sắp theo thứ tự tên file và các ảnh kề nhau cần có vùng chồng lấn.
- Đầu ra là panorama rộng hơn, cùng các ảnh trung gian, mask và số liệu đánh giá.
- Pipeline giả định quan hệ giữa từng cặp ảnh kề nhau có thể được mô tả đủ tốt bằng một homography.
- Kích thước ảnh gốc được giữ nguyên; không resize để tránh thay đổi hệ tọa độ keypoint.

## 2. Luồng xử lý

1. Đọc ảnh màu, tạo ảnh xám.
2. Phát hiện keypoint và tính descriptor bằng SIFT hoặc ORB.
3. So khớp từng cặp kề nhau bằng BFMatcher KNN; lọc bằng Lowe ratio test.
4. Dùng các correspondence để ước lượng homography với RANSAC.
5. Hợp thành các homography về ảnh tham chiếu, tính canvas chung và warp ảnh cùng mask.
6. Trộn vùng overlap bằng feather blending có xét mask.
7. Lưu panorama, mask, ảnh trực quan hóa và CSV.

SIFT dùng descriptor số thực và khoảng cách L2; ORB dùng descriptor nhị phân và khoảng cách Hamming. Các bước matching trở đi được gọi qua cùng API và cùng cấu hình pipeline để so sánh công bằng.

## 3. Nguyên lý và chỉ số sử dụng

Lowe ratio test giữ match khi khoảng cách tốt nhất $d_1$ nhỏ hơn ngưỡng $	au$ nhân khoảng cách thứ hai $d_2$: $d_1 < 	au d_2$. Cấu hình mặc định dùng $	au=0.75$.

Homography ánh xạ điểm ảnh nguồn sang ảnh đích: $lambda mathbf{x}_d = Hmathbf{x}_s$. RANSAC đánh giá các phép chiếu theo ngưỡng sai số reprojection 4 pixel. Tỷ lệ inlier là số inlier chia cho số good matches; reprojection error là sai số pixel trung bình trên các inlier.

Các chỉ số này đo độ nhất quán hình học nội bộ. Bộ ảnh không có ground-truth panorama nên không thể xem chúng là sai số tuyệt đối. Chất lượng cuối còn được xem trực quan qua đường nối, ghosting, biến dạng phối cảnh và mask vùng hợp lệ.

## 4. Khởi tạo môi trường và xác định project root

In [ ]:
from pathlib import Path
import sys

# Tìm project root từ thư mục hiện tại hoặc các thư mục cha.
_here = Path.cwd().resolve()
NOTEBOOK_PROJECT_ROOT = next(
    (
        parent for parent in (_here, *_here.parents)
        if (parent / "input" / "boat").is_dir()
        and (parent / "output").is_dir()
    ),
    None,
)
if NOTEBOOK_PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy project root. Mở notebook trong thư mục chứa input/boat."
    )
if str(NOTEBOOK_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_PROJECT_ROOT))

print("Project:", NOTEBOOK_PROJECT_ROOT)

## 5. Mã nguồn theo kiến trúc module

Mỗi module giữ một trách nhiệm riêng. Chạy các cell theo thứ tự; module sau sử dụng các lớp và hàm đã định nghĩa ở cell trước. Đây là mã thuật toán đầy đủ, không phải bản minh họa rút gọn.

## Module: src/blending.py

**Tác dụng:** Giữ pixel vùng chỉ có một ảnh; feather blend vùng overlap và lưu panorama/mask/alpha.

Cell kế tiếp chứa mã nguồn đầy đủ của module.

In [ ]:
"""Mask-aware feather blending for images on a shared canvas."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

import cv2
import numpy as np
from numpy.typing import NDArray


@dataclass(frozen=True)
class BlendResult:
    """Blended BGR panorama and masks describing its pixel coverage."""

    panorama: NDArray[np.uint8]
    mask: NDArray[np.uint8]
    overlap_mask: NDArray[np.uint8]
    coverage_count: NDArray[np.uint16]


def blend_images(
    images: Sequence[NDArray[np.uint8]],
    masks: Sequence[NDArray[np.uint8]],
) -> BlendResult:
    """Blend valid pixels, feathering only where two or more masks overlap.

    Images and masks may differ in size; they are top-left aligned and padded
    to the largest height and width. Pixels supported by only one mask are
    copied exactly. Empty masks contribute no pixels, and black image values
    remain valid wherever their mask says they are valid.
    """

    if len(images) != len(masks):
        raise ValueError(f"images and masks must have equal lengths; got {len(images)} and {len(masks)}.")
    if not images:
        raise ValueError("At least one image and mask pair is required.")

    checked_images: list[NDArray[np.uint8]] = []
    checked_masks: list[NDArray[np.bool_]] = []
    max_height = 0
    max_width = 0
    for index, (image, mask) in enumerate(zip(images, masks, strict=True)):
        if not isinstance(image, np.ndarray):
            raise TypeError(f"images[{index}] must be a NumPy array.")
        if image.ndim != 3 or image.shape[2] != 3 or image.dtype != np.uint8:
            raise ValueError(f"images[{index}] must be a uint8 BGR image with shape (H, W, 3).")
        if not isinstance(mask, np.ndarray):
            raise TypeError(f"masks[{index}] must be a NumPy array.")
        if mask.ndim == 3 and mask.shape[2] == 1:
            mask = mask[:, :, 0]
        if mask.ndim != 2:
            raise ValueError(f"masks[{index}] must be a 2-D mask; got shape {mask.shape}.")
        if mask.shape != image.shape[:2]:
            raise ValueError(
                f"masks[{index}] shape {mask.shape} does not match image shape {image.shape[:2]}."
            )
        checked_images.append(image)
        checked_masks.append(mask > 0)
        max_height = max(max_height, image.shape[0])
        max_width = max(max_width, image.shape[1])

    if max_height == 0 or max_width == 0:
        raise ValueError("Input images must have non-zero width and height.")

    coverage = np.zeros((max_height, max_width), dtype=np.uint16)
    for image, valid in zip(checked_images, checked_masks, strict=True):
        height, width = image.shape[:2]
        coverage[:height, :width] += valid.astype(np.uint16)

    output = np.zeros((max_height, max_width, 3), dtype=np.uint8)
    # Assign exclusive areas directly so their pixel values remain unchanged.
    for image, valid in zip(checked_images, checked_masks, strict=True):
        height, width = image.shape[:2]
        exclusive = valid & (coverage[:height, :width] == 1)
        target = output[:height, :width]
        target[exclusive] = image[exclusive]

    overlap = coverage >= 2
    if np.any(overlap):
        # Limit the floating-point accumulators to the overlap bounding box.
        # The panorama canvas can be much larger than its valid content.
        overlap_rows = np.flatnonzero(np.any(overlap, axis=1))
        overlap_columns = np.flatnonzero(np.any(overlap, axis=0))
        overlap_y0, overlap_y1 = int(overlap_rows[0]), int(overlap_rows[-1]) + 1
        overlap_x0, overlap_x1 = int(overlap_columns[0]), int(overlap_columns[-1]) + 1
        overlap_roi = overlap[overlap_y0:overlap_y1, overlap_x0:overlap_x1]
        roi_height, roi_width = overlap_roi.shape
        accumulated = np.zeros((roi_height, roi_width, 3), dtype=np.float32)
        total_weight = np.zeros((roi_height, roi_width), dtype=np.float32)

        for image, valid in zip(checked_images, checked_masks, strict=True):
            height, width = image.shape[:2]
            mask_rows = np.flatnonzero(np.any(valid, axis=1))
            mask_columns = np.flatnonzero(np.any(valid, axis=0))
            if mask_rows.size == 0 or mask_columns.size == 0:
                continue
            mask_y0, mask_y1 = int(mask_rows[0]), int(mask_rows[-1]) + 1
            mask_x0, mask_x1 = int(mask_columns[0]), int(mask_columns[-1]) + 1

            # Padding with one invalid pixel keeps distance-to-mask-edge values
            # equivalent to running the distance transform on the full canvas.
            local_valid = valid[mask_y0:mask_y1, mask_x0:mask_x1]
            padded_valid = cv2.copyMakeBorder(
                local_valid.astype(np.uint8), 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0
            )
            distance = cv2.distanceTransform(padded_valid, cv2.DIST_L2, 3)[1:-1, 1:-1]

            y0, y1 = max(overlap_y0, mask_y0), min(overlap_y1, mask_y1, height)
            x0, x1 = max(overlap_x0, mask_x0), min(overlap_x1, mask_x1, width)
            if y0 >= y1 or x0 >= x1:
                continue

            source_valid = valid[y0:y1, x0:x1]
            local_overlap = overlap[y0:y1, x0:x1]
            weights = np.maximum(
                distance[y0 - mask_y0 : y1 - mask_y0, x0 - mask_x0 : x1 - mask_x0],
                1.0,
            )
            weights *= source_valid & local_overlap

            target_y = slice(y0 - overlap_y0, y1 - overlap_y0)
            target_x = slice(x0 - overlap_x0, x1 - overlap_x0)
            source = image[y0:y1, x0:x1]
            # Channel-wise float32 operations avoid a full-canvas float64 RGB
            # temporary while preserving the same feather weighting rule.
            for channel in range(3):
                accumulated[target_y, target_x, channel] += (
                    source[:, :, channel].astype(np.float32) * weights
                )
            total_weight[target_y, target_x] += weights

        overlap_weights = total_weight[overlap_roi]
        if np.any(overlap_weights <= 0) or not np.isfinite(overlap_weights).all():
            raise RuntimeError("Feather blending produced invalid weights in an overlap region.")
        output_roi = output[overlap_y0:overlap_y1, overlap_x0:overlap_x1]
        blended = np.rint(accumulated[overlap_roi] / overlap_weights[:, None])
        output_roi[overlap_roi] = np.clip(blended, 0, 255).astype(np.uint8)

    valid_mask = (coverage > 0).astype(np.uint8) * 255
    overlap_mask = overlap.astype(np.uint8) * 255
    return BlendResult(
        panorama=output,
        mask=valid_mask,
        overlap_mask=overlap_mask,
        coverage_count=coverage,
    )


def save_blend_result(
    result: BlendResult,
    output_dir: str | Path,
    filename: str = "panorama_blended.png",
) -> list[Path]:
    """Save the blended panorama and its coverage/overlap masks."""

    directory = Path(output_dir)
    directory.mkdir(parents=True, exist_ok=True)
    rgba = cv2.cvtColor(result.panorama, cv2.COLOR_BGR2BGRA)
    rgba[:, :, 3] = result.mask
    artifacts = (
        (filename, result.panorama),
        ("panorama_transparent.png", rgba),
        ("panorama_valid_mask.png", result.mask),
        ("panorama_overlap_mask.png", result.overlap_mask),
    )
    saved: list[Path] = []
    for name, image in artifacts:
        path = directory / name
        if not cv2.imwrite(str(path), image):
            raise OSError(f"Could not save blending result: {path}")
        reopened = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if reopened is None or reopened.size == 0:
            raise OSError(f"Saved blending result could not be reopened: {path}")
        saved.append(path)
    return saved

## Module: src/config.py

**Tác dụng:** Khai báo project root, thư mục data/output và tham số riêng cho SIFT, ORB, matching, RANSAC.

Cell kế tiếp chứa mã nguồn đầy đủ của module.

In [ ]:
"""Project paths and shared settings for the image stitching pipeline."""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path


# The project root is the parent of ``src``. Resolving it here makes the
# defaults independent of the directory from which Python is launched.
PROJECT_ROOT = NOTEBOOK_PROJECT_ROOT


@dataclass(frozen=True)
class SIFTConfig:
    """Parameters passed to OpenCV's SIFT detector and descriptor."""

    nfeatures: int = 0
    n_octave_layers: int = 3
    contrast_threshold: float = 0.04
    edge_threshold: float = 10.0
    sigma: float = 1.6


@dataclass(frozen=True)
class ORBConfig:
    """Parameters passed to OpenCV's ORB detector and descriptor."""

    nfeatures: int = 2500
    scale_factor: float = 1.2
    n_levels: int = 8
    edge_threshold: int = 31
    first_level: int = 0
    wta_k: int = 2
    score_type: int = 0  # cv2.ORB_HARRIS_SCORE; kept numeric to avoid importing cv2 here.
    patch_size: int = 31
    fast_threshold: int = 20


@dataclass(frozen=True)
class MatchingConfig:
    """Shared feature matching settings; distance metric depends on descriptor."""

    ratio_threshold: float = 0.75
    minimum_matches: int = 8
    norm_name_sift: str = "NORM_L2"
    norm_name_orb: str = "NORM_HAMMING"


@dataclass(frozen=True)
class RansacConfig:
    """Settings and validity thresholds for homography estimation."""

    reprojection_threshold: float = 4.0
    max_iterations: int = 2000
    confidence: float = 0.995
    minimum_inliers: int = 8
    minimum_inlier_ratio: float = 0.2


@dataclass(frozen=True)
class ProjectConfig:
    """Paths and algorithm settings shared by the project."""

    project_root: Path = PROJECT_ROOT
    data_dir: Path = field(default_factory=lambda: PROJECT_ROOT / "input" / "boat")
    output_dir: Path = field(default_factory=lambda: PROJECT_ROOT / "output")
    sift: SIFTConfig = field(default_factory=SIFTConfig)
    orb: ORBConfig = field(default_factory=ORBConfig)
    matching: MatchingConfig = field(default_factory=MatchingConfig)
    ransac: RansacConfig = field(default_factory=RansacConfig)


DEFAULT_CONFIG = ProjectConfig()

## Module: src/evaluation.py

**Tác dụng:** Định nghĩa cấu trúc số liệu theo ảnh/cặp/lượt chạy và xuất thống kê CSV.

Cell kế tiếp chứa mã nguồn đầy đủ của module.

In [ ]:
"""Experiment metrics and CSV reporting for the stitching pipeline."""

from __future__ import annotations

import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence


@dataclass(frozen=True)
class ImageMetrics:
    """Feature count and extraction time for one input image."""

    index: int
    image_name: str
    keypoints: int
    feature_seconds: float


@dataclass(frozen=True)
class PairMetrics:
    """Matching and homography statistics for an adjacent image pair."""

    source_image: str
    destination_image: str
    knn_candidates: int
    good_matches: int
    inliers: int
    inlier_ratio: float
    reprojection_error: float
    matching_seconds: float
    homography_seconds: float


@dataclass(frozen=True)
class PipelineTimings:
    """Wall-clock timings for the main pipeline stages, in seconds."""

    image_loading: float
    feature_extraction: float
    matching: float
    homography: float
    warping: float
    blending: float
    total: float


@dataclass(frozen=True)
class PipelineResult:
    """Serializable experiment summary returned after a successful panorama."""

    method: str
    reference_image: str
    output_dir: Path
    image_metrics: tuple[ImageMetrics, ...]
    pair_metrics: tuple[PairMetrics, ...]
    timings: PipelineTimings
    canvas_width: int
    canvas_height: int
    artifacts: dict[str, Path]


@dataclass(frozen=True)
class PipelineFailure:
    """A method/pair failure recorded by the command-line runner."""

    method: str
    stage: str
    reason: str
    source_image: str = ""
    destination_image: str = ""


CSV_FIELDS = (
    "method",
    "record_type",
    "image_index",
    "image_name",
    "source_image",
    "destination_image",
    "reference_image",
    "keypoints",
    "source_keypoints",
    "destination_keypoints",
    "feature_seconds",
    "source_feature_seconds",
    "destination_feature_seconds",
    "knn_candidates",
    "good_matches",
    "inliers",
    "inlier_ratio",
    "reprojection_error_px",
    "matching_seconds",
    "homography_seconds",
    "image_loading_seconds",
    "feature_extraction_seconds_total",
    "matching_seconds_total",
    "homography_seconds_total",
    "warping_seconds",
    "blending_seconds",
    "pipeline_seconds_total",
    "canvas_width",
    "canvas_height",
    "success",
    "stage",
    "reason",
    "output_dir",
)


def write_evaluation_csv(
    results: Sequence[PipelineResult],
    output_path: str | Path,
    failures: Sequence[PipelineFailure] = (),
) -> Path:
    """Write image-, pair-, summary-, and failure-level records to a CSV file.

    The CSV contains measurements only. Human visual observations should be
    recorded separately in the report and are not inferred from match counts.
    """

    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows: list[dict[str, object]] = []

    for result in results:
        common = {
            "method": result.method,
            "success": True,
            "reference_image": result.reference_image,
            "output_dir": str(result.output_dir),
            "canvas_width": result.canvas_width,
            "canvas_height": result.canvas_height,
        }
        images_by_name = {metric.image_name: metric for metric in result.image_metrics}
        for metric in result.image_metrics:
            rows.append(
                {
                    **common,
                    "record_type": "image",
                    "image_index": metric.index,
                    "image_name": metric.image_name,
                    "keypoints": metric.keypoints,
                    "feature_seconds": metric.feature_seconds,
                }
            )
        for pair in result.pair_metrics:
            source_metric = images_by_name[pair.source_image]
            destination_metric = images_by_name[pair.destination_image]
            rows.append(
                {
                    **common,
                    "record_type": "pair",
                    "source_image": pair.source_image,
                    "destination_image": pair.destination_image,
                    "source_keypoints": source_metric.keypoints,
                    "destination_keypoints": destination_metric.keypoints,
                    "source_feature_seconds": source_metric.feature_seconds,
                    "destination_feature_seconds": destination_metric.feature_seconds,
                    "knn_candidates": pair.knn_candidates,
                    "good_matches": pair.good_matches,
                    "inliers": pair.inliers,
                    "inlier_ratio": pair.inlier_ratio,
                    "reprojection_error_px": pair.reprojection_error,
                    "matching_seconds": pair.matching_seconds,
                    "homography_seconds": pair.homography_seconds,
                }
            )
        timing = result.timings
        rows.append(
            {
                **common,
                "record_type": "summary",
                "image_loading_seconds": timing.image_loading,
                "feature_extraction_seconds_total": timing.feature_extraction,
                "matching_seconds_total": timing.matching,
                "homography_seconds_total": timing.homography,
                "warping_seconds": timing.warping,
                "blending_seconds": timing.blending,
                "pipeline_seconds_total": timing.total,
            }
        )

    for failure in failures:
        rows.append(
            {
                "method": failure.method,
                "record_type": "failure",
                "source_image": failure.source_image,
                "destination_image": failure.destination_image,
                "success": False,
                "stage": failure.stage,
                "reason": failure.reason,
            }
        )

    with path.open("w", encoding="utf-8-sig", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=CSV_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    return path

## Module: src/features.py

**Tác dụng:** Cung cấp một API để phát hiện keypoint và tính descriptor bằng SIFT hoặc ORB.

Cell kế tiếp chứa mã nguồn đầy đủ của module.

In [ ]:
"""Feature detection and description for SIFT and ORB."""

from __future__ import annotations

from typing import Any, Literal

import cv2
import numpy as np
from numpy.typing import NDArray


FeatureMethod = Literal["sift", "orb"]
GrayImage = NDArray[np.uint8]


def _normalize_method(method: str) -> FeatureMethod:
    normalized = method.strip().lower()
    if normalized not in {"sift", "orb"}:
        raise ValueError(f"Unsupported feature method {method!r}; use 'sift' or 'orb'.")
    return normalized  # type: ignore[return-value]


def extract_features(
    gray_image: GrayImage,
    method: FeatureMethod,
    config: SIFTConfig | ORBConfig | None = None,
) -> tuple[list[Any], NDArray[Any] | None]:
    """Detect keypoints and compute descriptors for one grayscale image.

    Args:
        gray_image: Non-empty, single-channel 8-bit image. Its dimensions and
            coordinate system are preserved.
        method: ``"sift"`` or ``"orb"`` (case-insensitive at runtime).
        config: Method-specific parameters. Defaults to ``DEFAULT_CONFIG``.

    Returns:
        A list of OpenCV keypoints and a descriptor matrix. If no features are
        found, returns an empty keypoint list and ``None`` descriptors.

    Raises:
        TypeError: If the image or method-specific config has the wrong type.
        ValueError: If the image has invalid dimensions or is empty.
        RuntimeError: If the selected detector is unavailable in this OpenCV build.
    """

    if not isinstance(gray_image, np.ndarray):
        raise TypeError("gray_image must be a NumPy array.")
    if gray_image.ndim != 2:
        raise ValueError(f"Expected a 2-D grayscale image; got shape {gray_image.shape}.")
    if gray_image.size == 0 or gray_image.shape[0] == 0 or gray_image.shape[1] == 0:
        raise ValueError("gray_image must not be empty.")
    if gray_image.dtype != np.uint8:
        raise TypeError(f"gray_image must have dtype uint8; got {gray_image.dtype}.")

    normalized = _normalize_method(method)
    if normalized == "sift":
        selected_config = DEFAULT_CONFIG.sift if config is None else config
        if not isinstance(selected_config, SIFTConfig):
            raise TypeError("SIFT extraction requires an SIFTConfig instance.")
        if not hasattr(cv2, "SIFT_create"):
            raise RuntimeError("This OpenCV build does not provide cv2.SIFT_create.")
        detector = cv2.SIFT_create(
            nfeatures=selected_config.nfeatures,
            nOctaveLayers=selected_config.n_octave_layers,
            contrastThreshold=selected_config.contrast_threshold,
            edgeThreshold=selected_config.edge_threshold,
            sigma=selected_config.sigma,
        )
    else:
        selected_config = DEFAULT_CONFIG.orb if config is None else config
        if not isinstance(selected_config, ORBConfig):
            raise TypeError("ORB extraction requires an ORBConfig instance.")
        detector = cv2.ORB_create(
            nfeatures=selected_config.nfeatures,
            scaleFactor=selected_config.scale_factor,
            nlevels=selected_config.n_levels,
            edgeThreshold=selected_config.edge_threshold,
            firstLevel=selected_config.first_level,
            WTA_K=selected_config.wta_k,
            scoreType=selected_config.score_type,
            patchSize=selected_config.patch_size,
            fastThreshold=selected_config.fast_threshold,
        )

    keypoints, descriptors = detector.detectAndCompute(gray_image, None)
    if not keypoints or descriptors is None or descriptors.size == 0:
        return [], None
    return list(keypoints), descriptors

## Module: src/homography.py

**Tác dụng:** Lấy cặp tọa độ tương ứng, ước lượng homography bằng RANSAC và thống kê inlier/reprojection error.

Cell kế tiếp chứa mã nguồn đầy đủ của module.

In [ ]:
"""Robust homography estimation and correspondence visualization."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import cv2
import numpy as np
from numpy.typing import NDArray



class HomographyEstimationError(RuntimeError):
    """Raised when correspondences cannot produce a valid homography."""


@dataclass(frozen=True)
class HomographyResult:
    """Homography mapping source-image coordinates into destination coordinates."""

    H: NDArray[np.float64]
    inlier_mask: NDArray[np.bool_]
    inlier_count: int
    inlier_ratio: float
    reprojection_error: float
    total_matches: int


def estimate_homography(
    keypoints_source: list[Any],
    keypoints_destination: list[Any],
    matches: list[Any],
    config: RansacConfig | None = None,
) -> HomographyResult:
    """Estimate the transform mapping source points into destination points.

    OpenCV ``DMatch.queryIdx`` indexes ``keypoints_source`` and ``trainIdx``
    indexes ``keypoints_destination``. RANSAC inlier thresholds and minimum
    acceptance criteria come from ``config.py``.

    Raises:
        TypeError: If inputs or config have an unexpected type.
        ValueError: If RANSAC parameters are invalid.
        HomographyEstimationError: If there are too few correspondences, RANSAC
            fails, or its output is degenerate or below configured thresholds.
    """

    if not isinstance(keypoints_source, (list, tuple)):
        raise TypeError("keypoints_source must be a list or tuple of OpenCV keypoints.")
    if not isinstance(keypoints_destination, (list, tuple)):
        raise TypeError("keypoints_destination must be a list or tuple of OpenCV keypoints.")
    if not isinstance(matches, (list, tuple)):
        raise TypeError("matches must be a list or tuple of OpenCV DMatch objects.")

    selected_config = DEFAULT_CONFIG.ransac if config is None else config
    if not isinstance(selected_config, RansacConfig):
        raise TypeError("config must be a RansacConfig instance.")
    if selected_config.reprojection_threshold <= 0:
        raise ValueError("reprojection_threshold must be greater than zero.")
    if selected_config.max_iterations <= 0:
        raise ValueError("max_iterations must be greater than zero.")
    if not 0.0 < selected_config.confidence < 1.0:
        raise ValueError("confidence must be greater than zero and less than one.")
    if selected_config.minimum_inliers < 4:
        raise ValueError("minimum_inliers must be at least four for a homography.")
    if not 0.0 <= selected_config.minimum_inlier_ratio <= 1.0:
        raise ValueError("minimum_inlier_ratio must be between zero and one.")

    total_matches = len(matches)
    if total_matches < 4:
        raise HomographyEstimationError(
            f"At least 4 point matches are required; received {total_matches}."
        )

    source_points = np.empty((total_matches, 2), dtype=np.float32)
    destination_points = np.empty((total_matches, 2), dtype=np.float32)
    for index, match in enumerate(matches):
        query_index = int(match.queryIdx)
        train_index = int(match.trainIdx)
        if query_index < 0 or query_index >= len(keypoints_source):
            raise HomographyEstimationError(
                f"Match {index} has queryIdx {query_index} outside the source keypoints."
            )
        if train_index < 0 or train_index >= len(keypoints_destination):
            raise HomographyEstimationError(
                f"Match {index} has trainIdx {train_index} outside the destination keypoints."
            )
        try:
            source_points[index] = keypoints_source[query_index].pt
            destination_points[index] = keypoints_destination[train_index].pt
        except (AttributeError, TypeError, ValueError) as exc:
            raise TypeError(f"Invalid keypoint or match at correspondence {index}.") from exc

    if not np.isfinite(source_points).all() or not np.isfinite(destination_points).all():
        raise HomographyEstimationError("Correspondence coordinates contain non-finite values.")

    H, cv_inlier_mask = cv2.findHomography(
        source_points,
        destination_points,
        method=cv2.RANSAC,
        ransacReprojThreshold=selected_config.reprojection_threshold,
        maxIters=selected_config.max_iterations,
        confidence=selected_config.confidence,
    )
    if H is None:
        raise HomographyEstimationError("cv2.findHomography returned no homography matrix.")
    if not isinstance(H, np.ndarray) or H.shape != (3, 3):
        raise HomographyEstimationError(
            f"Expected a 3x3 homography matrix; got {getattr(H, 'shape', None)}."
        )
    H = np.asarray(H, dtype=np.float64)
    if not np.isfinite(H).all():
        raise HomographyEstimationError("Homography matrix contains non-finite values.")

    scale = H[2, 2]
    if abs(scale) > np.finfo(np.float64).eps:
        H = H / scale
    else:
        norm = float(np.linalg.norm(H))
        if norm <= np.finfo(np.float64).eps:
            raise HomographyEstimationError("Homography matrix is the zero matrix.")
        H = H / norm
    if np.linalg.matrix_rank(H) < 3 or abs(float(np.linalg.det(H))) <= 1e-12:
        raise HomographyEstimationError("Estimated homography matrix is singular (degenerate).")

    if cv_inlier_mask is None:
        raise HomographyEstimationError("cv2.findHomography did not return an inlier mask.")
    inlier_mask = np.asarray(cv_inlier_mask).reshape(-1).astype(bool)
    if inlier_mask.size != total_matches:
        raise HomographyEstimationError(
            f"Inlier mask length {inlier_mask.size} does not match {total_matches} matches."
        )

    inlier_count = int(np.count_nonzero(inlier_mask))
    inlier_ratio = inlier_count / total_matches
    if inlier_count < selected_config.minimum_inliers:
        raise HomographyEstimationError(
            f"Only {inlier_count} inliers; at least {selected_config.minimum_inliers} required."
        )
    if inlier_ratio < selected_config.minimum_inlier_ratio:
        raise HomographyEstimationError(
            f"Inlier ratio {inlier_ratio:.3f} is below configured minimum "
            f"{selected_config.minimum_inlier_ratio:.3f}."
        )

    source_inliers = source_points[inlier_mask].reshape(-1, 1, 2)
    destination_inliers = destination_points[inlier_mask]
    projected = cv2.perspectiveTransform(source_inliers, H).reshape(-1, 2)
    if not np.isfinite(projected).all():
        raise HomographyEstimationError("Homography produced non-finite projected points.")
    errors = np.linalg.norm(projected - destination_inliers, axis=1)
    reprojection_error = float(np.mean(errors))
    if not np.isfinite(reprojection_error):
        raise HomographyEstimationError("Reprojection error is not finite.")

    return HomographyResult(
        H=H,
        inlier_mask=inlier_mask,
        inlier_count=inlier_count,
        inlier_ratio=inlier_ratio,
        reprojection_error=reprojection_error,
        total_matches=total_matches,
    )


def visualize_homography_matches(
    image_source: NDArray[np.uint8],
    keypoints_source: list[Any],
    image_destination: NDArray[np.uint8],
    keypoints_destination: list[Any],
    matches: list[Any],
    result: HomographyResult,
    method: FeatureMethod,
    *,
    output_dir: str | Path | None = None,
    filename: str = "homography_01_02.jpg",
) -> Path:
    """Save side-by-side correspondences (green inliers, red outliers).

    Output is written under ``output/sift`` or ``output/orb`` by default and
    reopened once to confirm the saved image is decodable.
    """

    normalized_method = _normalize_method(method)
    for name, image in (("image_source", image_source), ("image_destination", image_destination)):
        if not isinstance(image, np.ndarray):
            raise TypeError(f"{name} must be a NumPy array.")
        if image.ndim != 3 or image.shape[2] != 3 or image.size == 0:
            raise ValueError(f"{name} must be a non-empty 3-channel BGR image.")
        if image.dtype != np.uint8:
            raise TypeError(f"{name} must have dtype uint8.")
    if not isinstance(result, HomographyResult):
        raise TypeError("result must be a HomographyResult instance.")
    if len(matches) != result.total_matches or result.inlier_mask.size != len(matches):
        raise ValueError("Matches and homography inlier mask must have the same length.")
    if not filename or Path(filename).name != filename:
        raise ValueError("filename must be a non-empty file name without directory parts.")

    top_margin = 34
    canvas_height = top_margin + max(image_source.shape[0], image_destination.shape[0])
    canvas_width = image_source.shape[1] + image_destination.shape[1]
    canvas = np.zeros((canvas_height, canvas_width, 3), dtype=np.uint8)
    canvas[top_margin : top_margin + image_source.shape[0], : image_source.shape[1]] = image_source
    destination_x = image_source.shape[1]
    canvas[
        top_margin : top_margin + image_destination.shape[0],
        destination_x : destination_x + image_destination.shape[1],
    ] = image_destination

    cv2.putText(canvas, "Inliers: green", (12, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 220, 0), 2)
    cv2.putText(canvas, "Outliers: red", (190, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    for match, is_inlier in zip(matches, result.inlier_mask, strict=True):
        try:
            source_point = keypoints_source[int(match.queryIdx)].pt
            destination_point = keypoints_destination[int(match.trainIdx)].pt
        except (IndexError, AttributeError, TypeError, ValueError) as exc:
            raise ValueError("Match indices do not correspond to the supplied keypoints.") from exc
        color = (0, 220, 0) if is_inlier else (0, 0, 255)
        source_xy = (int(round(source_point[0])), int(round(source_point[1])) + top_margin)
        destination_xy = (
            int(round(destination_point[0])) + destination_x,
            int(round(destination_point[1])) + top_margin,
        )
        cv2.line(canvas, source_xy, destination_xy, color, 1, cv2.LINE_AA)
        cv2.circle(canvas, source_xy, 3, color, -1, cv2.LINE_AA)
        cv2.circle(canvas, destination_xy, 3, color, -1, cv2.LINE_AA)

    target_dir = Path(output_dir) if output_dir is not None else DEFAULT_CONFIG.output_dir / normalized_method
    target_dir.mkdir(parents=True, exist_ok=True)
    output_path = target_dir / filename
    if not cv2.imwrite(str(output_path), canvas):
        raise OSError(f"Could not save homography visualization: {output_path}")
    reopened = cv2.imread(str(output_path), cv2.IMREAD_COLOR)
    if reopened is None or reopened.size == 0:
        raise OSError(f"Saved homography visualization could not be reopened: {output_path}")
    return output_path

## Module: src/image_io.py

**Tác dụng:** Đọc ảnh theo thứ tự tên file, xác thực ảnh, giữ ảnh màu BGR và tạo ảnh xám; không resize.

Cell kế tiếp chứa mã nguồn đầy đủ của module.

In [ ]:
"""Image loading and preprocessing helpers for image stitching.

Images are returned in filename order. Their original dimensions are kept;
no resizing or other geometric transform is applied. Color arrays use OpenCV's
BGR channel order, and grayscale arrays are derived from those same pixels.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Final

import numpy as np



SUPPORTED_EXTENSIONS: Final[frozenset[str]] = frozenset(
    {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}
)


class ImageLoadError(RuntimeError):
    """Raised when an input directory or one of its images cannot be loaded."""


@dataclass(frozen=True)
class LoadedImage:
    """An input image and its grayscale view, with the source path retained."""

    path: Path
    color: np.ndarray
    gray: np.ndarray


def _get_cv2():
    """Import OpenCV lazily so callers receive an actionable dependency error."""

    try:
        import cv2
    except ImportError as exc:
        raise ImageLoadError(
            "OpenCV is required to load images. Install it with "
            "`python -m pip install opencv-python`."
        ) from exc
    return cv2


def load_images(
    directory: str | Path = DEFAULT_CONFIG.data_dir,
    *,
    extensions: frozenset[str] = SUPPORTED_EXTENSIONS,
) -> list[LoadedImage]:
    """Load supported images in case-insensitive filename order.

    Args:
        directory: Folder containing one panorama sequence.
        extensions: Supported filename suffixes, case-insensitive.

    Returns:
        Loaded images in deterministic filename order. Color images remain BGR
        and grayscale versions are generated for feature detection.

    Raises:
        ImageLoadError: If the folder is missing/not a directory, contains no
            supported images, OpenCV is unavailable, or a file cannot be read.
    """

    image_dir = Path(directory).expanduser()
    if not image_dir.exists():
        raise ImageLoadError(f"Image directory does not exist: {image_dir}")
    if not image_dir.is_dir():
        raise ImageLoadError(f"Image path is not a directory: {image_dir}")

    normalized_extensions = frozenset(
        extension.lower() if extension.startswith(".") else f".{extension.lower()}"
        for extension in extensions
    )
    paths = sorted(
        (
            path
            for path in image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in normalized_extensions
        ),
        key=lambda path: (path.name.casefold(), path.name),
    )
    if not paths:
        suffixes = ", ".join(sorted(normalized_extensions))
        raise ImageLoadError(
            f"No supported images found in {image_dir}. Supported extensions: {suffixes}"
        )

    cv2 = _get_cv2()
    loaded: list[LoadedImage] = []
    for path in paths:
        color = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if color is None or color.size == 0:
            raise ImageLoadError(f"Could not read image file: {path}")
        if color.ndim != 3 or color.shape[2] != 3:
            raise ImageLoadError(
                f"Expected a 3-channel color image at {path}; got shape {color.shape}"
            )

        gray = cv2.cvtColor(color, cv2.COLOR_BGR2GRAY)
        if gray is None or gray.size == 0:
            raise ImageLoadError(f"Could not convert image to grayscale: {path}")
        loaded.append(LoadedImage(path=path, color=color, gray=gray))

    return loaded

## Module: src/main.py

**Tác dụng:** Điểm vào chương trình: chọn phương pháp/dữ liệu, chạy SIFT và ORB, tổng hợp CSV và báo lỗi.

Cell kế tiếp chứa mã nguồn đầy đủ của module.

In [ ]:
"""Command-line entry point for running SIFT and ORB panorama pipelines."""

from __future__ import annotations

import argparse
from pathlib import Path
from typing import Sequence



def build_parser() -> argparse.ArgumentParser:
    """Build the CLI parser for data/output paths and feature methods."""

    parser = argparse.ArgumentParser(description="Run the custom image stitching pipeline.")
    parser.add_argument("--data-dir", type=Path, default=DEFAULT_CONFIG.data_dir)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_CONFIG.output_dir)
    parser.add_argument(
        "--methods",
        nargs="+",
        choices=("sift", "orb"),
        default=("sift", "orb"),
        help="Feature methods to run; both are used by default.",
    )
    parser.add_argument(
        "--reference-index",
        type=int,
        default=None,
        help="Reference image index (default: central image in filename order).",
    )
    return parser


def main(argv: Sequence[str] | None = None) -> int:
    """Run selected methods, report pair failures, and write aggregate CSV."""

    args = build_parser().parse_args(argv)
    results: list[PipelineResult] = []
    failures: list[PipelineFailure] = []
    for method in args.methods:
        try:
            result = run_pipeline(
                method,
                image_dir=args.data_dir,
                output_root=args.output_dir,
                reference_index=args.reference_index,
            )
        except PipelineError as exc:
            failures.append(
                PipelineFailure(
                    method=exc.method,
                    stage=exc.stage,
                    reason=exc.reason,
                    source_image=exc.source_image,
                    destination_image=exc.destination_image,
                )
            )
            print(f"FAILED: {exc}")
            continue
        except Exception as exc:
            failures.append(
                PipelineFailure(method=method, stage="setup", reason=str(exc))
            )
            print(f"FAILED: {method.upper()} setup: {exc}")
            continue

        results.append(result)
        print(
            f"{method.upper()}: panorama={result.artifacts['panorama_blended']} "
            f"size={result.canvas_width}x{result.canvas_height}; "
            f"total={result.timings.total:.3f}s"
        )
        for image in result.image_metrics:
            print(f"  {image.image_name}: {image.keypoints} keypoints ({image.feature_seconds:.3f}s)")
        for pair in result.pair_metrics:
            print(
                f"  {pair.source_image}-{pair.destination_image}: "
                f"good={pair.good_matches}, inliers={pair.inliers} "
                f"({pair.inlier_ratio:.3f}), reprojection={pair.reprojection_error:.3f}px"
            )

    aggregate_csv = write_evaluation_csv(
        results, args.output_dir / "evaluation.csv", failures=failures
    )
    print(f"Evaluation CSV: {aggregate_csv}")
    return 1 if failures else 0

## Module: src/matching.py

**Tác dụng:** So khớp descriptor bằng BFMatcher KNN, lọc Lowe ratio test và lưu hình minh họa matches.

Cell kế tiếp chứa mã nguồn đầy đủ của module.

In [ ]:
"""Descriptor matching and visualization for SIFT and ORB features."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

import cv2
import numpy as np
from numpy.typing import NDArray



@dataclass(frozen=True)
class MatchStatistics:
    """Counts and status explaining the output of a descriptor match."""

    query_descriptors: int
    train_descriptors: int
    knn_candidates: int
    good_matches: int
    minimum_required: int
    status: str

    @property
    def has_enough_matches(self) -> bool:
        """Whether the good match count meets the configured minimum."""

        return self.good_matches >= self.minimum_required


@dataclass(frozen=True)
class MatchResult:
    """Ratio-filtered matches and their summary statistics."""

    matches: list[Any]
    statistics: MatchStatistics


def match_descriptors(
    descriptors_query: NDArray[Any] | None,
    descriptors_train: NDArray[Any] | None,
    method: FeatureMethod,
    config: MatchingConfig | None = None,
) -> MatchResult:
    """Match descriptors using BFMatcher, KNN (k=2), and Lowe's ratio test.

    Missing or too-short descriptor sets return an empty result with a status
    instead of raising. Malformed arrays and invalid configuration raise errors.
    """

    normalized = _normalize_method(method)
    selected_config = DEFAULT_CONFIG.matching if config is None else config
    if not isinstance(selected_config, MatchingConfig):
        raise TypeError("config must be a MatchingConfig instance.")
    if not 0.0 < selected_config.ratio_threshold < 1.0:
        raise ValueError("ratio_threshold must be greater than 0 and less than 1.")
    if selected_config.minimum_matches < 0:
        raise ValueError("minimum_matches must not be negative.")

    query_count = 0 if descriptors_query is None else len(descriptors_query)
    train_count = 0 if descriptors_train is None else len(descriptors_train)

    def empty_result(candidate_count: int, status: str) -> MatchResult:
        return MatchResult(
            matches=[],
            statistics=MatchStatistics(
                query_descriptors=query_count,
                train_descriptors=train_count,
                knn_candidates=candidate_count,
                good_matches=0,
                minimum_required=selected_config.minimum_matches,
                status=status,
            ),
        )

    if descriptors_query is None or descriptors_train is None:
        return empty_result(0, "missing_descriptors")
    for name, descriptors in (
        ("descriptors_query", descriptors_query),
        ("descriptors_train", descriptors_train),
    ):
        if not isinstance(descriptors, np.ndarray):
            raise TypeError(f"{name} must be a NumPy array or None.")
        if descriptors.ndim != 2:
            raise ValueError(f"{name} must be a 2-D array; got shape {descriptors.shape}.")
        if descriptors.shape[1] == 0:
            raise ValueError(f"{name} must have at least one descriptor element.")

    if query_count == 0:
        return empty_result(0, "empty_query_descriptors")
    if train_count < 2:
        return empty_result(0, "insufficient_train_descriptors_for_knn")

    if normalized == "sift":
        if descriptors_query.dtype not in (np.dtype(np.float32), np.dtype(np.uint8)):
            raise TypeError("SIFT descriptors must have dtype float32 (or supported uint8).")
        if descriptors_train.dtype != descriptors_query.dtype:
            raise TypeError("Query and train SIFT descriptors must have the same dtype.")
        norm_type = getattr(cv2, selected_config.norm_name_sift, None)
        if norm_type is None:
            raise ValueError(f"Unknown OpenCV norm: {selected_config.norm_name_sift}")
    else:
        if descriptors_query.dtype != np.uint8 or descriptors_train.dtype != np.uint8:
            raise TypeError("ORB descriptors must have dtype uint8.")
        norm_type = getattr(cv2, selected_config.norm_name_orb, None)
        if norm_type is None:
            raise ValueError(f"Unknown OpenCV norm: {selected_config.norm_name_orb}")
        if descriptors_query.shape[1] != descriptors_train.shape[1]:
            raise ValueError("Query and train descriptors must have the same width.")

    matcher = cv2.BFMatcher(normType=norm_type, crossCheck=False)
    knn_matches = matcher.knnMatch(descriptors_query, descriptors_train, k=2)
    good_matches = [
        pair[0]
        for pair in knn_matches
        if len(pair) >= 2 and pair[0].distance < selected_config.ratio_threshold * pair[1].distance
    ]
    status = "ok" if len(good_matches) >= selected_config.minimum_matches else "insufficient_good_matches"
    if not good_matches:
        status = "no_good_matches"

    return MatchResult(
        matches=good_matches,
        statistics=MatchStatistics(
            query_descriptors=query_count,
            train_descriptors=train_count,
            knn_candidates=sum(1 for pair in knn_matches if len(pair) >= 2),
            good_matches=len(good_matches),
            minimum_required=selected_config.minimum_matches,
            status=status,
        ),
    )


def visualize_matches(
    image_query: NDArray[np.uint8],
    keypoints_query: list[Any],
    image_train: NDArray[np.uint8],
    keypoints_train: list[Any],
    result: MatchResult,
    method: FeatureMethod,
    *,
    output_dir: str | Path | None = None,
    filename: str = "matches_01_02.jpg",
) -> Path:
    """Draw and save ratio-filtered matches; return the saved image path.

    The default output location is ``output/sift`` or ``output/orb``.
    Inputs are expected to be BGR color images, such as ``LoadedImage.color``.
    """

    normalized = _normalize_method(method)
    for name, image in (("image_query", image_query), ("image_train", image_train)):
        if not isinstance(image, np.ndarray):
            raise TypeError(f"{name} must be a NumPy array.")
        if image.ndim != 3 or image.shape[2] != 3 or image.size == 0:
            raise ValueError(f"{name} must be a non-empty 3-channel BGR image.")
        if image.dtype != np.uint8:
            raise TypeError(f"{name} must have dtype uint8.")
    if not isinstance(result, MatchResult):
        raise TypeError("result must be a MatchResult instance.")
    if not filename or Path(filename).name != filename:
        raise ValueError("filename must be a non-empty file name without directory parts.")

    target_dir = Path(output_dir) if output_dir is not None else DEFAULT_CONFIG.output_dir / normalized
    target_dir.mkdir(parents=True, exist_ok=True)
    output_path = target_dir / filename
    visualization = cv2.drawMatches(
        image_query,
        keypoints_query,
        image_train,
        keypoints_train,
        result.matches,
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )
    if visualization is None or visualization.size == 0:
        raise RuntimeError("OpenCV did not produce a match visualization.")
    if not cv2.imwrite(str(output_path), visualization):
        raise OSError(f"Could not save match visualization: {output_path}")
    reopened = cv2.imread(str(output_path), cv2.IMREAD_COLOR)
    if reopened is None or reopened.size == 0:
        raise OSError(f"Saved visualization could not be reopened: {output_path}")
    return output_path

## Module: src/pipeline.py

**Tác dụng:** Điều phối trọn pipeline cho một phương pháp; dừng có thông báo nếu matching hoặc homography thất bại.

Cell kế tiếp chứa mã nguồn đầy đủ của module.

In [ ]:
"""End-to-end orchestration for one image feature method."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
from typing import Any

import cv2
import numpy as np



class PipelineError(RuntimeError):
    """A stage failure with method and adjacent-pair context."""

    def __init__(
        self,
        method: str,
        stage: str,
        reason: str,
        source_image: str = "",
        destination_image: str = "",
    ) -> None:
        self.method = method
        self.stage = stage
        self.reason = reason
        self.source_image = source_image
        self.destination_image = destination_image
        pair = f" pair {source_image}–{destination_image}" if source_image else ""
        super().__init__(f"{method.upper()}{pair} failed during {stage}: {reason}")


def _save_unblended_preview(images: list[Any], output_path: Path) -> Path:
    """Save a first-valid-pixel composite to show the seam before blending."""

    if not images:
        raise ValueError("At least one warped image is required for a preview.")
    height, width = images[0].image.shape[:2]
    preview = np.zeros((height, width, 3), dtype=np.uint8)
    occupied = np.zeros((height, width), dtype=bool)
    for item in images:
        valid = item.mask > 0
        take = valid & ~occupied
        preview[take] = item.image[take]
        occupied |= valid
    if not cv2.imwrite(str(output_path), preview):
        raise OSError(f"Could not save pre-blending preview: {output_path}")
    reopened = cv2.imread(str(output_path), cv2.IMREAD_COLOR)
    if reopened is None or reopened.size == 0:
        raise OSError(f"Saved pre-blending preview could not be reopened: {output_path}")
    return output_path


def _pair_label(source: LoadedImage, destination: LoadedImage) -> str:
    return f"{source.path.stem}_{destination.path.stem}"


def run_pipeline(
    method: FeatureMethod,
    image_dir: str | Path = DEFAULT_CONFIG.data_dir,
    output_root: str | Path = DEFAULT_CONFIG.output_dir,
    reference_index: int | None = None,
) -> PipelineResult:
    """Run the existing load→features→matches→homographies→warp→blend stages.

    Each input image's descriptors are computed once. Only adjacent image pairs
    are matched and estimated. Pairwise homographies retain the established
    source-to-destination convention and are composed by ``warping.py``.
    Any match or homography failure aborts this method before panorama creation.
    """

    normalized_method = _normalize_method(method)
    pipeline_start = perf_counter()
    load_start = perf_counter()
    images = load_images(image_dir)
    loading_seconds = perf_counter() - load_start
    if len(images) < 2:
        raise PipelineError(normalized_method, "image loading", "At least two images are required.")
    if reference_index is None:
        reference_index = (len(images) - 1) // 2
    if not 0 <= reference_index < len(images):
        raise ValueError(f"reference_index must be in [0, {len(images) - 1}].")

    method_output_dir = Path(output_root) / normalized_method
    method_output_dir.mkdir(parents=True, exist_ok=True)

    keypoints_by_image: list[list[Any]] = []
    descriptors_by_image: list[np.ndarray | None] = []
    image_metrics: list[ImageMetrics] = []
    feature_total = 0.0
    for index, image in enumerate(images):
        start = perf_counter()
        try:
            keypoints, descriptors = extract_features(image.gray, normalized_method)  # type: ignore[arg-type]
        except (ValueError, TypeError, RuntimeError) as exc:
            raise PipelineError(normalized_method, "feature extraction", f"{image.path.name}: {exc}") from exc
        elapsed = perf_counter() - start
        feature_total += elapsed
        if descriptors is None or not keypoints:
            raise PipelineError(
                normalized_method,
                "feature extraction",
                f"No descriptors found in {image.path.name}.",
            )
        keypoints_by_image.append(keypoints)
        descriptors_by_image.append(descriptors)
        image_metrics.append(ImageMetrics(index, image.path.name, len(keypoints), elapsed))

    homographies: list[np.ndarray] = []
    pair_metrics: list[PairMetrics] = []
    matching_total = 0.0
    homography_total = 0.0
    for index in range(len(images) - 1):
        source = images[index]
        destination = images[index + 1]
        label = _pair_label(source, destination)
        match_start = perf_counter()
        try:
            match_result = match_descriptors(
                descriptors_by_image[index],
                descriptors_by_image[index + 1],
                normalized_method,  # type: ignore[arg-type]
            )
        except (ValueError, TypeError, cv2.error) as exc:
            raise PipelineError(
                normalized_method, "matching", str(exc), source.path.name, destination.path.name
            ) from exc
        matching_seconds = perf_counter() - match_start
        matching_total += matching_seconds
        if not match_result.statistics.has_enough_matches:
            raise PipelineError(
                normalized_method,
                "matching",
                f"{match_result.statistics.good_matches} good matches; "
                f"{match_result.statistics.minimum_required} required "
                f"(status={match_result.statistics.status}).",
                source.path.name,
                destination.path.name,
            )

        homography_start = perf_counter()
        try:
            homography = estimate_homography(
                keypoints_by_image[index], keypoints_by_image[index + 1], match_result.matches
            )
        except (HomographyEstimationError, ValueError, TypeError, cv2.error) as exc:
            raise PipelineError(
                normalized_method, "homography", str(exc), source.path.name, destination.path.name
            ) from exc
        homography_seconds = perf_counter() - homography_start
        homography_total += homography_seconds

        try:
            visualize_matches(
                source.color,
                keypoints_by_image[index],
                destination.color,
                keypoints_by_image[index + 1],
                match_result,
                normalized_method,  # type: ignore[arg-type]
                output_dir=method_output_dir,
                filename=f"matches_{label}.jpg",
            )
            visualize_homography_matches(
                source.color,
                keypoints_by_image[index],
                destination.color,
                keypoints_by_image[index + 1],
                match_result.matches,
                homography,
                normalized_method,  # type: ignore[arg-type]
                output_dir=method_output_dir,
                filename=f"homography_{label}.jpg",
            )
        except (ValueError, TypeError, OSError, cv2.error) as exc:
            raise PipelineError(
                normalized_method, "visualization", str(exc), source.path.name, destination.path.name
            ) from exc

        homographies.append(homography.H)
        pair_metrics.append(
            PairMetrics(
                source_image=source.path.name,
                destination_image=destination.path.name,
                knn_candidates=match_result.statistics.knn_candidates,
                good_matches=match_result.statistics.good_matches,
                inliers=homography.inlier_count,
                inlier_ratio=homography.inlier_ratio,
                reprojection_error=homography.reprojection_error,
                matching_seconds=matching_seconds,
                homography_seconds=homography_seconds,
            )
        )

    warp_start = perf_counter()
    try:
        warped = warp_images_to_reference(
            [image.color for image in images], homographies, reference_index=reference_index
        )
        save_warped_images(warped, method_output_dir)
    except (ValueError, RuntimeError, OSError, cv2.error) as exc:
        raise PipelineError(normalized_method, "warping", str(exc)) from exc
    warping_seconds = perf_counter() - warp_start

    blending_start = perf_counter()
    try:
        _save_unblended_preview(
            warped.images, method_output_dir / "panorama_before_blending.png"
        )
        blended = blend_images(
            [item.image for item in warped.images], [item.mask for item in warped.images]
        )
        save_blend_result(blended, method_output_dir)
    except (ValueError, RuntimeError, OSError, cv2.error) as exc:
        raise PipelineError(normalized_method, "blending", str(exc)) from exc
    blending_seconds = perf_counter() - blending_start
    total_seconds = perf_counter() - pipeline_start

    artifacts = {
        "panorama_before_blending": method_output_dir / "panorama_before_blending.png",
        "panorama_blended": method_output_dir / "panorama_blended.png",
        "panorama_transparent": method_output_dir / "panorama_transparent.png",
        "valid_mask": method_output_dir / "panorama_valid_mask.png",
        "overlap_mask": method_output_dir / "panorama_overlap_mask.png",
    }
    for index in range(len(images)):
        artifacts[f"warped_image_{index:02d}"] = method_output_dir / f"warped_{index:02d}_image.png"
        artifacts[f"warped_mask_{index:02d}"] = method_output_dir / f"warped_{index:02d}_mask.png"
    for index in range(len(pair_metrics)):
        label = f"{images[index].path.stem}_{images[index + 1].path.stem}"
        artifacts[f"matches_{label}"] = method_output_dir / f"matches_{label}.jpg"
        artifacts[f"homography_{label}"] = method_output_dir / f"homography_{label}.jpg"

    result = PipelineResult(
        method=normalized_method,
        reference_image=images[reference_index].path.name,
        output_dir=method_output_dir,
        image_metrics=tuple(image_metrics),
        pair_metrics=tuple(pair_metrics),
        timings=PipelineTimings(
            image_loading=loading_seconds,
            feature_extraction=feature_total,
            matching=matching_total,
            homography=homography_total,
            warping=warping_seconds,
            blending=blending_seconds,
            total=total_seconds,
        ),
        canvas_width=warped.canvas_size[0],
        canvas_height=warped.canvas_size[1],
        artifacts=artifacts,
    )
    write_evaluation_csv([result], method_output_dir / "evaluation.csv")
    return result

## Module: src/warping.py

**Tác dụng:** Hợp thành homography ảnh kề nhau về ảnh tham chiếu, tính canvas chung rồi warp ảnh và mask.

Cell kế tiếp chứa mã nguồn đầy đủ của module.

In [ ]:
"""Project images into a common canvas using source-to-destination homographies."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

import cv2
import numpy as np
from numpy.typing import NDArray


class WarpingError(RuntimeError):
    """Raised when image transforms cannot be safely projected to a canvas."""


@dataclass(frozen=True)
class WarpedImage:
    """One color image and its valid-pixel mask on the shared panorama canvas."""

    image: NDArray[np.uint8]
    mask: NDArray[np.uint8]
    source_to_reference: NDArray[np.float64]
    source_to_canvas: NDArray[np.float64]


@dataclass(frozen=True)
class WarpedPanorama:
    """All images warped to one common canvas, before blending."""

    images: list[WarpedImage]
    reference_index: int
    canvas_size: tuple[int, int]  # (width, height)
    canvas_translation: NDArray[np.float64]
    bounds: tuple[int, int, int, int]  # (min_x, min_y, max_x, max_y), inclusive


def _validated_homography(matrix: NDArray[np.generic], label: str) -> NDArray[np.float64]:
    """Return a normalized, finite, invertible 3x3 matrix."""

    value = np.asarray(matrix, dtype=np.float64)
    if value.shape != (3, 3):
        raise ValueError(f"{label} must have shape (3, 3); got {value.shape}.")
    if not np.isfinite(value).all():
        raise ValueError(f"{label} contains non-finite values.")
    scale = value[2, 2]
    if abs(scale) > np.finfo(np.float64).eps:
        value = value / scale
    else:
        norm = float(np.linalg.norm(value))
        if norm <= np.finfo(np.float64).eps:
            raise ValueError(f"{label} is the zero matrix.")
        value = value / norm
    if np.linalg.matrix_rank(value) < 3 or abs(float(np.linalg.det(value))) <= 1e-12:
        raise ValueError(f"{label} is singular and cannot be used for warping.")
    return value


def transforms_to_reference(
    image_count: int,
    pairwise_homographies: Sequence[NDArray[np.generic]],
    reference_index: int = 0,
) -> list[NDArray[np.float64]]:
    """Compose adjacent transforms into source-image-to-reference transforms.

    Each ``pairwise_homographies[i]`` maps coordinates in image ``i`` to
    coordinates in image ``i + 1``. If ``H_i`` maps i→i+1 and ``T_i`` maps i
    to the reference, composition uses ``T_i = T_(i+1) @ H_i`` to the left of
    the reference and ``T_(i+1) = T_i @ inverse(H_i)`` to its right.
    """

    if image_count < 1:
        raise ValueError("image_count must be at least one.")
    if not 0 <= reference_index < image_count:
        raise ValueError(f"reference_index must be in [0, {image_count - 1}].")
    if len(pairwise_homographies) != image_count - 1:
        raise ValueError(
            f"Expected {image_count - 1} adjacent homographies for {image_count} images; "
            f"received {len(pairwise_homographies)}."
        )

    adjacent = [
        _validated_homography(matrix, f"pairwise_homographies[{index}]")
        for index, matrix in enumerate(pairwise_homographies)
    ]
    transforms: list[NDArray[np.float64] | None] = [None] * image_count
    transforms[reference_index] = np.eye(3, dtype=np.float64)

    # H_i maps i to i+1, so left-compose it onto the transform already mapping
    # image i+1 into the reference coordinate system.
    for index in range(reference_index - 1, -1, -1):
        transforms[index] = transforms[index + 1] @ adjacent[index]  # type: ignore[operator]

    # To travel from image i+1 back to i, invert H_i, then apply T_i.
    for index in range(reference_index, image_count - 1):
        try:
            inverse = np.linalg.inv(adjacent[index])
        except np.linalg.LinAlgError as exc:
            raise WarpingError(f"Adjacent homography {index} could not be inverted.") from exc
        transforms[index + 1] = transforms[index] @ inverse  # type: ignore[operator]

    return [
        _validated_homography(transform, f"transform for image {index}")
        for index, transform in enumerate(transforms)
        if transform is not None
    ]


def _project_corners(
    image: NDArray[np.uint8], transform: NDArray[np.float64], index: int
) -> NDArray[np.float64]:
    """Project pixel-center corners, rejecting a projective horizon crossing."""

    height, width = image.shape[:2]
    corners = np.array(
        [[0, 0, 1], [width - 1, 0, 1], [width - 1, height - 1, 1], [0, height - 1, 1]],
        dtype=np.float64,
    ).T
    projected_h = transform @ corners
    denominators = projected_h[2]
    if np.any(np.abs(denominators) <= 1e-10) or (
        np.min(denominators) < 0 < np.max(denominators)
    ):
        raise WarpingError(f"Image {index} transform crosses the projective horizon.")
    projected = (projected_h[:2] / denominators).T
    if not np.isfinite(projected).all():
        raise WarpingError(f"Image {index} has non-finite projected corners.")
    return projected


def warp_images_to_reference(
    images: Sequence[NDArray[np.uint8]],
    pairwise_homographies: Sequence[NDArray[np.generic]],
    reference_index: int = 0,
    *,
    max_canvas_pixels: int = 100_000_000,
) -> WarpedPanorama:
    """Warp BGR images and validity masks onto a shared reference canvas.

    The pairwise matrices must map image i to image i+1, matching
    ``HomographyResult.H``. The reference image is left unchanged apart from
    the common canvas translation needed to handle negative coordinates.
    """

    if not isinstance(images, (list, tuple)) or not images:
        raise ValueError("images must be a non-empty list or tuple.")
    if max_canvas_pixels <= 0:
        raise ValueError("max_canvas_pixels must be greater than zero.")
    checked_images: list[NDArray[np.uint8]] = []
    for index, image in enumerate(images):
        if not isinstance(image, np.ndarray):
            raise TypeError(f"images[{index}] must be a NumPy array.")
        if image.ndim != 3 or image.shape[2] != 3 or image.size == 0:
            raise ValueError(f"images[{index}] must be a non-empty 3-channel BGR image.")
        if image.dtype != np.uint8:
            raise TypeError(f"images[{index}] must have dtype uint8.")
        checked_images.append(image)

    reference_transforms = transforms_to_reference(
        len(checked_images), pairwise_homographies, reference_index
    )
    all_corners = [
        _project_corners(image, transform, index)
        for index, (image, transform) in enumerate(zip(checked_images, reference_transforms, strict=True))
    ]
    corners = np.concatenate(all_corners, axis=0)
    min_x, min_y = np.floor(corners.min(axis=0)).astype(np.int64)
    max_x, max_y = np.ceil(corners.max(axis=0)).astype(np.int64)
    width = int(max_x - min_x + 1)
    height = int(max_y - min_y + 1)
    if width <= 0 or height <= 0:
        raise WarpingError(f"Computed invalid canvas size {(width, height)}.")
    if width * height > max_canvas_pixels:
        raise WarpingError(
            f"Canvas {width}x{height} ({width * height:,} pixels) exceeds limit "
            f"{max_canvas_pixels:,}. Check homographies or raise max_canvas_pixels."
        )

    translation = np.array(
        [[1.0, 0.0, -float(min_x)], [0.0, 1.0, -float(min_y)], [0.0, 0.0, 1.0]],
        dtype=np.float64,
    )
    canvas_images: list[WarpedImage] = []
    for image, reference_transform in zip(checked_images, reference_transforms, strict=True):
        source_to_canvas = translation @ reference_transform
        warped_image = cv2.warpPerspective(
            image,
            source_to_canvas,
            (width, height),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=(0, 0, 0),
        )
        source_mask = np.full(image.shape[:2], 255, dtype=np.uint8)
        warped_mask = cv2.warpPerspective(
            source_mask,
            source_to_canvas,
            (width, height),
            flags=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=0,
        )
        canvas_images.append(
            WarpedImage(
                image=warped_image,
                mask=warped_mask,
                source_to_reference=reference_transform,
                source_to_canvas=source_to_canvas,
            )
        )

    return WarpedPanorama(
        images=canvas_images,
        reference_index=reference_index,
        canvas_size=(width, height),
        canvas_translation=translation,
        bounds=(int(min_x), int(min_y), int(max_x), int(max_y)),
    )


def save_warped_images(warped: WarpedPanorama, output_dir: str | Path) -> list[Path]:
    """Save each warped BGR image and its valid-pixel mask as PNG files."""

    directory = Path(output_dir)
    directory.mkdir(parents=True, exist_ok=True)
    saved: list[Path] = []
    for index, item in enumerate(warped.images):
        for suffix, array in (("image", item.image), ("mask", item.mask)):
            path = directory / f"warped_{index:02d}_{suffix}.png"
            if not cv2.imwrite(str(path), array):
                raise OSError(f"Could not save warped {suffix}: {path}")
            reopened = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
            if reopened is None or reopened.size == 0:
                raise OSError(f"Saved warped {suffix} could not be reopened: {path}")
            saved.append(path)
    return saved

## 6. Cấu hình lần chạy

Mặc định dùng bộ boat và chạy cả hai phương pháp. Có thể đổi DATA_DIR sang thư mục ảnh khác; tên file phải giữ thứ tự chụp. Kết quả được tách dưới output/notebook để không ghi đè output/sift, output/orb hoặc CSV cũ.

In [ ]:
# Mặc định giữ nguyên cấu hình hiện tại của project.
# Có thể đổi DATA_DIR thành thư mục khác chứa chuỗi ảnh theo thứ tự tên.
DATA_DIR = DEFAULT_CONFIG.data_dir
OUTPUT_ROOT = DEFAULT_CONFIG.output_dir / "notebook"
METHODS = ("sift", "orb")

print("Dữ liệu:", DATA_DIR)
print("Output:", OUTPUT_ROOT)
print("Phương pháp:", METHODS)

## 7. Load và tiền xử lý ảnh

Cell này kiểm tra số lượng ảnh, thứ tự file và kích thước màu/xám, sau đó hiển thị đầu vào. Nếu bộ dữ liệu không có 3–4 ảnh, notebook đưa cảnh báo để người dùng kiểm tra lại.

In [ ]:
from IPython.display import HTML, Image, display

def display_bgr(image, title, max_width=900):
    height, width = image.shape[:2]
    if width > max_width:
        new_height = round(height * max_width / width)
        image = cv2.resize(image, (max_width, new_height), interpolation=cv2.INTER_AREA)
    ok, encoded = cv2.imencode(".jpg", image, [cv2.IMWRITE_JPEG_QUALITY, 88])
    if not ok:
        raise RuntimeError(f"Could not encode preview: {title}")
    print(f"{title} — {width}x{height}")
    display(Image(data=encoded.tobytes()))

loaded_images = load_images(DATA_DIR)
print("Thứ tự ảnh:", [item.path.name for item in loaded_images])
if len(loaded_images) not in (3, 4):
    print(f"Cảnh báo: đề bài yêu cầu 3–4 ảnh; hiện có {len(loaded_images)} ảnh.")
for item in loaded_images:
    print(f"{item.path.name}: màu={item.color.shape}, xám={item.gray.shape}")
    display_bgr(item.color, item.path.name)

## 8. Chạy cùng pipeline cho SIFT và ORB

Cell dưới gọi entry point main được giữ từ module mã nguồn. Mỗi ảnh chỉ được trích xuất đặc trưng một lần cho mỗi phương pháp; các cặp ảnh kề nhau được matching và ước lượng homography. Nếu một cặp lỗi, thông báo và dòng failure được ghi nhận thay vì âm thầm tạo panorama.

In [ ]:
arguments = [
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_ROOT),
    "--methods", *METHODS,
]
exit_code = main(arguments)
if exit_code:
    print("Có lỗi ở một hoặc nhiều phương pháp. Xem thông báo và bản ghi failure trong CSV.")
else:
    print("Pipeline hoàn tất cho các phương pháp đã chọn.")

## 9. Đánh giá định lượng

CSV được tách thành bản ghi theo ảnh, theo cặp và tổng pipeline. Số keypoint cho biết số điểm được phát hiện; good matches và inlier mô tả matching/homography; thời gian được tính bằng giây. Không kết luận phương pháp tốt hơn chỉ từ tổng số matches.

In [ ]:
import csv
from html import escape

evaluation_path = OUTPUT_ROOT / "evaluation.csv"
with evaluation_path.open("r", newline="", encoding="utf-8-sig") as stream:
    evaluation_rows = list(csv.DictReader(stream))

def show_table(title, rows, columns):
    print(title)
    header = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(row.get(column, '') or '')}</td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(
        "<style>table{border-collapse:collapse;margin-bottom:12px}"
        "th,td{border:1px solid #aaa;padding:4px;text-align:left;font-size:11px}</style>"
        f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"
    ))

show_table(
    "Keypoint theo ảnh",
    [row for row in evaluation_rows if row["record_type"] == "image"],
    ["method", "image_name", "keypoints", "feature_seconds"],
)
show_table(
    "Matching và homography theo cặp kề nhau",
    [row for row in evaluation_rows if row["record_type"] == "pair"],
    ["method", "source_image", "destination_image", "good_matches", "inliers",
     "inlier_ratio", "reprojection_error_px", "matching_seconds", "homography_seconds"],
)
show_table(
    "Thời gian và kích thước panorama",
    [row for row in evaluation_rows if row["record_type"] == "summary"],
    ["method", "image_loading_seconds", "feature_extraction_seconds_total",
     "matching_seconds_total", "homography_seconds_total", "warping_seconds",
     "blending_seconds", "pipeline_seconds_total", "canvas_width", "canvas_height"],
)
failures = [row for row in evaluation_rows if row["record_type"] == "failure"]
if failures:
    show_table("Các lỗi được ghi nhận", failures, ["method", "stage", "source_image", "destination_image", "reason"])
else:
    print("Không có bản ghi failure trong lượt chạy này.")
print("CSV:", evaluation_path)

## 10. Kiểm tra trực quan panorama, blending và các cặp

Với từng phương pháp, xem panorama trước/sau blending, mask hợp lệ, overlap mask và ảnh matching/inlier-outlier của tất cả cặp kề nhau. Chú ý đường nối, vùng mờ/ghosting, phần cảnh bị biến dạng và vùng canvas trống.

In [ ]:
for method in METHODS:
    method_dir = OUTPUT_ROOT / method
    print()
    print("##", method.upper())
    for filename, title in [
        ("panorama_before_blending.png", "Panorama trước blending"),
        ("panorama_blended.png", "Panorama sau blending"),
        ("panorama_valid_mask.png", "Mask vùng hợp lệ"),
        ("panorama_overlap_mask.png", "Mask vùng overlap"),
    ]:
        artifact = method_dir / filename
        if artifact.is_file():
            image = cv2.imread(str(artifact), cv2.IMREAD_COLOR)
            if image is not None:
                display_bgr(image, title)
    for index in range(len(loaded_images) - 1):
        source = loaded_images[index].path.stem
        destination = loaded_images[index + 1].path.stem
        label = f"{source}_{destination}"
        for prefix, title in [
            ("matches", "Matches sau ratio test"),
            ("homography", "Inlier/outlier sau RANSAC"),
        ]:
            artifact = method_dir / f"{prefix}_{label}.jpg"
            if artifact.is_file():
                image = cv2.imread(str(artifact), cv2.IMREAD_COLOR)
                if image is not None:
                    display_bgr(image, f"{title}: {source}–{destination}")

## 11. Nhận xét và so sánh

Khi viết nhận xét, kết hợp số liệu và quan sát ảnh:

- **Độ chính xác căn chỉnh:** xem tỷ lệ inlier và reprojection error theo từng cặp; không có ground truth nên đây là bằng chứng hình học gián tiếp.
- **Tính liên tục:** kiểm tra ranh giới cảnh, đường nối, ghosting và việc các chi tiết bị lặp hoặc đứt.
- **Chất lượng trực quan:** xem panorama SIFT/ORB ở cùng tỷ lệ và đối chiếu mask.
- **Hiệu năng:** so thời gian từng giai đoạn; kết quả thời gian phụ thuộc máy, độ phân giải và lượt chạy.
- **Thất bại:** nếu có bản ghi failure, nêu cặp ảnh, giai đoạn và nguyên nhân. Nếu không có thì chỉ kết luận các cặp của bộ đang chạy đã thành công.

Không suy ra chất lượng panorama chỉ từ số matches hoặc tỷ lệ inlier. Feather blending làm chuyển tiếp màu dịu hơn nhưng không sửa parallax, chuyển động hay sai lệch hình học.

## 12. Thử nghiệm tùy chọn trên bộ ảnh thứ hai

Trong project có thêm input/nature. Bật cờ RUN_SECOND_DATASET để chạy chính entry point, cùng SIFT/ORB và cùng pipeline trên bộ đó. Đây là lượt chạy riêng; kết quả được lưu tách để không trộn CSV hoặc ảnh với bộ boat.

In [ ]:
RUN_SECOND_DATASET = False
SECOND_DATA_DIR = NOTEBOOK_PROJECT_ROOT / "input" / "nature"
SECOND_OUTPUT_ROOT = OUTPUT_ROOT / "nature"

if RUN_SECOND_DATASET:
    if not SECOND_DATA_DIR.is_dir():
        raise FileNotFoundError(f"Không tìm thấy bộ dữ liệu thứ hai: {SECOND_DATA_DIR}")
    second_exit_code = main([
        "--data-dir", str(SECOND_DATA_DIR),
        "--output-dir", str(SECOND_OUTPUT_ROOT),
        "--methods", *METHODS,
    ])
    print("Mã kết thúc lượt chạy bộ thứ hai:", second_exit_code)
else:
    print("Chưa bật thử nghiệm thứ hai. Đặt RUN_SECOND_DATASET = True nếu muốn chạy input/nature.")

## 13. Giới hạn và tái lập

- Dùng OpenCV, NumPy và Jupyter/IPython trong interpreter được chọn làm kernel.
- Chạy notebook từ trên xuống; dùng Restart Kernel and Run All để tránh biến cũ trong bộ nhớ.
- Data mặc định: input/boat. Output chính: output/notebook.
- Nếu đổi bộ ảnh, bảo đảm thứ tự tên đúng thứ tự chụp và các cặp kề nhau có overlap.
- Canvas chữ nhật có thể chứa vùng ngoài mask; panorama alpha và valid mask giúp xác định phần ảnh hợp lệ.
- Homography toàn cục và feather blending là mô hình cơ bản; không có bundle adjustment, seam optimization hoặc exposure compensation.
- Thư mục tests/ còn theo kiến trúc cũ và import src; cần chuyển đổi riêng trước khi chạy lại các test sau khi src đã được gộp vào notebook.